# GAVE2 Four-Output Ensemble on Google Colab

This notebook runs the full-resolution GAVE2 pipeline with four outputs:

- `cmrrwnet`
- `sam3`
- `yolo_native`
- `ensemble`

It assumes your Google Drive contains:

```text
MyDrive/MICCAI2026/
  GAVE2_preliminary/
  experiments/
  knowledge_base/   # optional, but useful for official CMRRWNet fallback
```

By the end, Drive will contain branch outputs under `MyDrive/MICCAI2026/submissions/`.

## 0. Runtime Choice

In Colab, choose:

`Runtime` -> `Change runtime type` -> `GPU`

Recommended hardware:

- A100 40GB: train full branch/fold runs.
- L4/T4: smoke tests, prediction, validation, or smaller `base_channels`.

This notebook keeps images at native `1536 x 1024`; it does not crop or resize the dataset.

In [ ]:
#@title Configuration
from pathlib import Path

PROJECT_NAME = "MICCAI2026"
TEAM_ID = "team_id"  # change this to your official team ID before final zip
DRIVE_PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
WORKDIR = Path("/content") / PROJECT_NAME
DATA_ROOT = WORKDIR / "GAVE2_preliminary"
RUN_DIR = DRIVE_PROJECT / "runs" / "gave2_ensemble"
SUBMISSION_ROOT = DRIVE_PROJECT / "submissions"
OOF_ROOT = DRIVE_PROJECT / "submissions_oof"

# Recommended A100 defaults. For quick experiments use 16 or 32.
BASE_CHANNELS = 64
EPOCHS_FULL = 150  # max cap; early stopping usually ends earlier
FOLDS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 1
EARLY_STOPPING_PATIENCE = 25
EARLY_STOPPING_MIN_DELTA = 1e-4
WORKERS = 2
AMP = "bf16"
PREPROCESS = "gray_clahe"
LOSS_MODE = "official_bce3"


# If you uploaded one zip to MyDrive/MICCAI2026, leave ZIP_NAME empty to auto-detect.
# If there are multiple zips, set it exactly, e.g. ZIP_NAME = "MICCAI2026.zip".
ZIP_NAME = "miccai.zip"
CLEAN_WORKDIR = True
print("Drive project:", DRIVE_PROJECT)
print("Workdir:", WORKDIR)
print("Run dir:", RUN_DIR)
print("Submission root:", SUBMISSION_ROOT)

In [ ]:
#@title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title Unpack Project Zip Or Copy Folders From Drive
import shutil
import zipfile
from pathlib import Path

WORKDIR.mkdir(parents=True, exist_ok=True)

if CLEAN_WORKDIR and WORKDIR.exists():
    for child in WORKDIR.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

extract_dir = Path("/content/_miccai2026_zip_extract")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

zip_candidates = sorted(DRIVE_PROJECT.glob("*.zip"))
selected_zip = None
if ZIP_NAME:
    selected_zip = DRIVE_PROJECT / ZIP_NAME
    if not selected_zip.exists():
        raise FileNotFoundError(f"ZIP_NAME was set but not found: {selected_zip}")
elif zip_candidates:
    preferred = [
        DRIVE_PROJECT / "miccai.zip",
        DRIVE_PROJECT / "MICCAI2026.zip",
        DRIVE_PROJECT / "miccai2026.zip",
        DRIVE_PROJECT / "GAVE2_preliminary.zip",
        DRIVE_PROJECT / "gave2_preliminary.zip",
        DRIVE_PROJECT / "gave2_colab.zip",
    ]
    selected_zip = next((p for p in preferred if p.exists()), zip_candidates[0])


def copy_dir(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"copied {src} -> {dst}")


def find_named_dir(root: Path, name: str):
    direct = root / name
    if direct.is_dir():
        return direct
    matches = [p for p in root.rglob(name) if p.is_dir()]
    return matches[0] if matches else None


def find_dataset_root(root: Path):
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for p in candidates:
        if (p / "training").is_dir() and (p / "validation").is_dir():
            return p
    return None

if selected_zip is not None:
    print("Using zip:", selected_zip)
    with zipfile.ZipFile(selected_zip) as zf:
        zf.extractall(extract_dir)

    dataset_src = find_named_dir(extract_dir, "GAVE2_preliminary") or find_dataset_root(extract_dir)
    if dataset_src is None:
        raise FileNotFoundError("Could not find GAVE2_preliminary or training/validation folders inside zip")
    copy_dir(dataset_src, WORKDIR / "GAVE2_preliminary")

    for name in ["experiments", "knowledge_base"]:
        src = find_named_dir(extract_dir, name)
        if src is not None:
            copy_dir(src, WORKDIR / name)
        elif (DRIVE_PROJECT / name).is_dir():
            copy_dir(DRIVE_PROJECT / name, WORKDIR / name)
        else:
            print(f"optional folder not found: {name}")
else:
    print("No zip found. Falling back to folder copy from Drive.")
    for name in ["GAVE2_preliminary", "experiments", "knowledge_base"]:
        src = DRIVE_PROJECT / name
        if src.is_dir():
            copy_dir(src, WORKDIR / name)
        elif name == "knowledge_base":
            print("optional folder not found: knowledge_base")
        else:
            raise FileNotFoundError(f"Required folder not found and no zip available: {src}")

if not (WORKDIR / "experiments" / "gave2_ensemble").is_dir():
    raise FileNotFoundError(
        "experiments/gave2_ensemble was not found. Include the experiments folder in the zip, "
        "or keep MyDrive/MICCAI2026/experiments as a Drive folder."
    )

print("Ready workspace:", WORKDIR)
print("Dataset:", WORKDIR / "GAVE2_preliminary")
print("Experiment code:", WORKDIR / "experiments" / "gave2_ensemble")

In [ ]:
#@title Install / Check Dependencies
%%bash
set -e
python -m pip install -q --upgrade pip
python -m pip install -q numpy pillow
python - <<'PY'
import sys
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(props.total_memory / 1024**3, 2))
PY

In [ ]:
#@title Verify Dataset Shape And Package Imports
import sys
sys.path.insert(0, str(WORKDIR))

from experiments.gave2_ensemble.data import GAVE2Dataset

train = GAVE2Dataset(DATA_ROOT, "training", "task2")
val = GAVE2Dataset(DATA_ROOT, "validation", "task2", require_target=False)
train_sample = train[0]
val_sample = val[0]

print("Training cases:", len(train), train_sample.case_id, train_sample.image.shape, train_sample.target.shape)
print("Validation cases:", len(val), val_sample.case_id, val_sample.image.shape, val_sample.mask.shape)
assert train_sample.image.shape == (5, 1024, 1536)
assert train_sample.target.shape == (3, 1024, 1536)
assert val_sample.image.shape == (5, 1024, 1536)

## 1. Smoke Tests

Run this section before long training. It trains each branch for one epoch on two cases, only to check tensor shapes, memory, checkpoints, and basic runtime.

In [ ]:
#@title Smoke Test: Task 2, All Three Branches
%%bash
set -e
cd /content/MICCAI2026
COMMON="--task task2 --data-root GAVE2_preliminary --out-dir /content/drive/MyDrive/MICCAI2026/runs/gave2_ensemble_smoke --epochs 1 --folds 2 --limit-cases 2 --base-channels 16 --batch-size 1 --grad-accum 1 --amp bf16 --workers 2 --early-stopping-patience 5 --early-stopping-metric best_dice --preprocess gray_clahe --loss-mode official_bce3"
python -m experiments.gave2_ensemble.train --branch cmrrwnet ${COMMON}
python -m experiments.gave2_ensemble.train --branch sam3 ${COMMON}
python -m experiments.gave2_ensemble.train --branch yolo_native ${COMMON}

## 2. Full Training

Run one fold at a time in Colab. This is safer because Colab sessions can disconnect.

Start with Task 2 because it has 40% leaderboard weight and uses CFP + FFA. Then train Task 1.

In [ ]:
#@title Train One Fold
# Change these and rerun the cell.
BRANCH = "cmrrwnet"   # cmrrwnet, sam3, yolo_native
TASK = "task2"        # task2 first, then task1
FOLD = 0              # 0, 1, 2, 3, 4

!cd {WORKDIR} && python -m experiments.gave2_ensemble.train \
  --branch {BRANCH} \
  --task {TASK} \
  --data-root {DATA_ROOT} \
  --out-dir {RUN_DIR} \
  --epochs {EPOCHS_FULL} \
  --folds {FOLDS} \
  --fold {FOLD} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --base-channels {BASE_CHANNELS} \
  --num-iterations 5 \
  --amp {AMP} \
  --workers {WORKERS} \
  --preprocess {PREPROCESS} \
  --loss-mode {LOSS_MODE} \
  --early-stopping-patience {EARLY_STOPPING_PATIENCE} \
  --early-stopping-min-delta {EARLY_STOPPING_MIN_DELTA} \
  --early-stopping-metric best_dice

In [ ]:
#@title Check Which Checkpoints Exist
from pathlib import Path

for branch in ["cmrrwnet", "sam3", "yolo_native"]:
    for task in ["task1", "task2"]:
        ckpts = sorted((RUN_DIR / branch / task).glob("fold_*/best.pt"))
        print(branch, task, len(ckpts), "best checkpoints")
        for p in ckpts:
            print("  ", p)

## 3. Prediction For Individual Branches

After each branch has checkpoints, generate Task 1 and Task 2 validation outputs. These are saved directly to Drive.

In [ ]:
#@title Predict One Branch
PRED_BRANCH = "cmrrwnet"  # cmrrwnet, sam3, yolo_native

for task in ["task1", "task2"]:
    !cd {WORKDIR} && python -m experiments.gave2_ensemble.predict \
      --branch {PRED_BRANCH} \
      --task {task} \
      --data-root {DATA_ROOT} \
      --run-dir {RUN_DIR} \
      --output-root {SUBMISSION_ROOT} \
      --team-id {TEAM_ID} \
      --tta flips \
      --preprocess auto \
      --workers {WORKERS}

In [ ]:
#@title Generate Task 3 For One Branch
TASK3_BRANCH = "cmrrwnet"  # cmrrwnet, sam3, yolo_native

!cd {WORKDIR} && python -m experiments.gave2_ensemble.biomarkers \
  --branch {TASK3_BRANCH} \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID}

## 4. Ensemble Output

First use the default weights: CMRRWNet `0.45`, SAM3-native `0.35`, YOLO-native `0.20`.

Optional: if you generated out-of-fold training predictions, run the weight optimization section below.

In [ ]:
#@title Create Default Ensemble And Task 3
!cd {WORKDIR} && python -m experiments.gave2_ensemble.ensemble \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID}

!cd {WORKDIR} && python -m experiments.gave2_ensemble.biomarkers \
  --branch ensemble \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID}

## 5. Optional Leakage-Safe Ensemble Weight Tuning

Use this only after all folds are trained. It creates out-of-fold predictions on training cases, then grid-searches per-channel weights.

In [ ]:
#@title Generate Out-Of-Fold Predictions For Weight Tuning
for branch in ["cmrrwnet", "sam3", "yolo_native"]:
    !cd {WORKDIR} && python -m experiments.gave2_ensemble.predict_oof \
      --branch {branch} \
      --task task2 \
      --data-root {DATA_ROOT} \
      --run-dir {RUN_DIR} \
      --output-root {OOF_ROOT} \
      --team-id {TEAM_ID} \
      --tta flips \
      --preprocess auto \
      --workers {WORKERS}

In [ ]:
#@title Optimize Per-Channel Ensemble Weights And Rebuild Ensemble
WEIGHTS_JSON = RUN_DIR / "ensemble_weights_task2.json"

!cd {WORKDIR} && python -m experiments.gave2_ensemble.optimize_ensemble_weights \
  --data-root {DATA_ROOT} \
  --submission-root {OOF_ROOT} \
  --team-id {TEAM_ID} \
  --task-name Task2 \
  --out {WEIGHTS_JSON}

!cd {WORKDIR} && python -m experiments.gave2_ensemble.ensemble \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID} \
  --weights-json {WEIGHTS_JSON}

!cd {WORKDIR} && python -m experiments.gave2_ensemble.biomarkers \
  --branch ensemble \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID}

## 6. Validate Four Outputs

The validator checks all four branches for 50 Task 1 PNGs, 50 Task 2 PNGs, and 50 Task 3 TXT files, with exact `1536 x 1024` RGB probability PNGs.

In [ ]:
#@title Validate All Submission Folders
!cd {WORKDIR} && python -m experiments.gave2_ensemble.validate_outputs \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID} \
  --height 1024 \
  --width 1536

In [ ]:
#@title Zip One Branch For Upload
# Change ZIP_BRANCH to cmrrwnet, sam3, yolo_native, or ensemble.
ZIP_BRANCH = "ensemble"
ZIP_PATH = DRIVE_PROJECT / f"{ZIP_BRANCH}_{TEAM_ID}.zip"

!cd {SUBMISSION_ROOT / ZIP_BRANCH} && zip -qr {ZIP_PATH} {TEAM_ID}
print("Wrote", ZIP_PATH)

## Exercise

Run the prediction section for all three individual branches, then run the ensemble section.

Answer scaffold:

```text
cmrrwnet validation complete: yes/no
sam3 validation complete: yes/no
yolo_native validation complete: yes/no
ensemble validation complete: yes/no
best preliminary upload candidate: ________
```

## Pitfalls And Extensions

Common pitfall: training all folds in one Colab session can waste progress if Colab disconnects. Prefer one branch/fold per cell execution, and save checkpoints to Drive.

Current Task 3 note: the implemented Task 3 generator is a deterministic proxy from Task 2 probability maps and ROI masks. It gives complete TXT files for testing, but the competitive next step is optic-disc-aware Zone C caliber extraction and validation-fold calibration.

Useful extension: after you get leaderboard feedback for each of the four outputs, we can tune ensemble weights using the public score pattern and add a stronger biomarker calibrator.